# Netflix Churn Baseline
В этом ноутбуке мы выполняем второе задание: построение и оценка бейзлайновых моделей.
- Константная модель
- Простая бейзлайновая модель (логистическая регрессия)

In [ ]:
from google.colab import files
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Загрузка данных
uploaded = files.upload()
df = pd.read_csv('netflix_user_behavior_dataset.csv')
print(df.head())

In [ ]:
# Целевой признак
le_target = LabelEncoder()
y = le_target.fit_transform(df['churned'])
print("Распределение churned:")
print(pd.Series(y).value_counts())

In [ ]:
# Признаки
categorical_cols = ['gender', 'country', 'subscription_type', 'payment_method', 'primary_device', 'favorite_genre']
numerical_cols = ['age', 'account_age_months', 'monthly_fee', 'devices_used', 'avg_watch_time_minutes',
                  'watch_sessions_per_week', 'binge_watch_sessions', 'completion_rate', 'rating_given',
                  'content_interactions', 'recommendation_click_rate', 'days_since_last_login']

X = df[categorical_cols + numerical_cols].copy()
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

In [ ]:
# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Константная бейзлайновая модель
constant_model = DummyClassifier(strategy='most_frequent', random_state=42)
constant_model.fit(X_train, y_train)
y_pred_constant = constant_model.predict(X_test)
f1_const = f1_score(y_test, y_pred_constant, zero_division=0)
print('F1-score константной модели:', f1_const)

In [ ]:
# Логистическая регрессия
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

logreg = LogisticRegression(random_state=42, max_iter=500, class_weight='balanced')
logreg.fit(X_train_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_scaled)
f1_logreg = f1_score(y_test, y_pred_logreg, zero_division=0)
print('F1-score логистической регрессии:', f1_logreg)